# Tutorial 7-3: Naïve Bayes and the Zero-Frequency Fix
**Course:** CSEN 140: Machine Learning/Data Mining  
**Instructor:** Dr. David C. Anastasiu

--- 

## 1. The Zero-Frequency Problem
Naïve Bayes relies on the assumption of conditional independence. To predict a class $Y$, we multiply the prior probability $P(Y)$ by the conditional probabilities of each attribute $P(X_i | Y)$. 

Mathematically, this product is highly sensitive. If a specific attribute value (e.g., `Marital Status = Married`) never appeared with a certain class (e.g., `Evade = Yes`) in our training data, then $P(X_i | Y) = 0$. Because we are multiplying these probabilities together, that single zero will force the entire final probability to zero, regardless of how strong the other evidence is. 

**Objective:** In this tutorial, we will:
1. Calculate probabilities for a sample containing a "novel" attribute value.
2. Implement **Laplace Smoothing** to prevent probability collapse.
3. Implement the **m-estimate** to handle classes with very low sample counts.

In [ ]:
import pandas as pd
import numpy as np

# Create a small training set inspired by the slides
data = {
    'Refund': ['Yes', 'No', 'No', 'Yes', 'No', 'No', 'Yes', 'No', 'No', 'No'],
    'Marital': ['Single', 'Married', 'Single', 'Married', 'Divorced', 'Married', 'Divorced', 'Single', 'Married', 'Single'],
    'Evade': ['No', 'No', 'No', 'No', 'Yes', 'No', 'No', 'Yes', 'No', 'Yes']
}
df = pd.DataFrame(data)

print("Training Data:")
print(df)

## 2. Demonstrating the Collapse
Imagine a test record: `(Refund = Yes, Marital = Married)`. 

Let's look at $P(Refund=Yes | Evade=Yes)$. In our data, every time someone evaded taxes, their refund status was 'No'. Therefore, the probability of 'Yes' given 'Yes' is 0.

In [ ]:
# Filter for Class = Yes
evaders = df[df['Evade'] == 'Yes']
n_evaders = len(evaders)

# Calculate probability of Refund=Yes given Evade=Yes
count_refund_yes = len(evaders[evaders['Refund'] == 'Yes'])
p_raw = count_refund_yes / n_evaders

print(f"Raw P(Refund=Yes | Evade=Yes): {p_raw}")
print(f"This will cause the entire Naive Bayes product for 'Evade=Yes' to be 0.")

## 3. The Laplace Fix
Laplace smoothing adds 1 to the numerator and the number of possible attribute values (c) to the denominator:
$$P(A_i | C) = \frac{N_{ic} + 1}{N_c + c}$$

This ensures that every outcome has at least a tiny, non-zero probability.

In [ ]:
c = df['Refund'].nunique()  # Number of possible values for Refund (Yes, No)
p_laplace = (count_refund_yes + 1) / (n_evaders + c)

print(f"Laplace Smoothed P(Refund=Yes | Evade=Yes): {p_laplace:.4f}")

## 4. The m-estimate Fix
The m-estimate is more flexible. It uses a prior probability $p$ and a parameter $m$ (the "equivalent sample size"):
$$P(A_i | C) = \frac{N_{ic} + mp}{N_c + m}$$

If $m=0$, we get the raw frequency. As $m$ increases, we trust our prior $p$ more than the small sample size in our data.

In [ ]:
m = 3 # Parameter
p_prior = 1/2 # Assuming equal prior for Yes/No refund

p_m_est = (count_refund_yes + m * p_prior) / (n_evaders + m)
print(f"m-estimate P(Refund=Yes | Evade=Yes): {p_m_est:.4f}")

## Summary
Without smoothing, Naïve Bayes is fragile. Even with millions of records, one missing combination can break the model. Laplace smoothing and m-estimates provide the "safety net" needed for robust real-world classification.